<a href="https://colab.research.google.com/github/LeandroHCarvalho/agentes-2026-2-equipe-agentes_especiais/blob/main/TemplateTrabalho.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Equipe Agentes Especiais

### Integrantes:
* Beatriz
* Lara
* Leandro



### Nosso problema em uma frase:

Hoje, quem vai alugar um imóvel precisa entender rapidamente todas as obrigações, prazos e taxas do contrato, mas os documentos possuem linguagem jurídica complexa e requisitos técnicos extensos, o que causa insegurança ao assinar e surpresas financeiras indesejadas durante a locação.

### Quem sofre com isso?
Pessoas que precisam alugar uma moradia.

### Como se resolve hoje?
Hoje quem precisa alugar uma moradia, fica a mercê, pois os contratos tem linguagem juridica complexa e falta uma compreensão melhor sobre as legislações vigentes. Quem tem acesso a um advogado consegue se resguardar de alguns abusos, mas a grande maioria das pessoas, simplesmente assinam o contrato mesmo com insegurança, pois precisam de um local para morar.

## PEAS







Analisando a complexidade do problema que nos xxxxx a resolver, optamos por utilizar o Workflow + Base Vetorial (RAG).

##Porque o RAG?

Vamos utilizar o RAG porque vamos utilizar as legislações vigentes no brasil a respeito de inquilinato e leis referentes presentes no Código Civil e também no Código de Defesa do Consumidor.

In [1]:
# Instalação das dependências do projeto
!pip install -q openai python-dotenv pydantic tenacity pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 97.0 MB/s eta 0:00:00




---



openai

Biblioteca oficial da OpenAI.

python-dotenv

Usada para gerenciar variáveis de ambiente e segredos de forma segura. Ela carrega chaves de API (como a OPENAI_API_KEY) a partir de um arquivo oculto chamado .env, evitando que você exponha suas chaves diretamente no código.

pydantic

Biblioteca de validação de dados e estruturação de objetos baseada em tipos do Python. É amplamente utilizada em projetos de IA para garantir que as respostas fornecidas pelos modelos sigam um formato exato (como um JSON bem definido) através de Structured Outputs.

tenacity

Biblioteca usada para implementar mecânicas de tentativa e erro (retry). Muito útil para chamadas de API, pois permite reconectar ou tentar novamente caso haja falhas de rede, taxas limite atingidas (rate limits) ou indisponibilidade temporária do serviço.

pdfplumber

Ferramenta para extrair texto, tabelas e dados visuais de arquivos PDF. Muito comum em projetos que precisam ler documentos PDF para alimentar modelos de linguagem ou pipelines de RAG (Retrieval-Augmented Generation).

Resumo do projeto
Pelas bibliotecas escolhidas, você provavelmente está desenvolvendo uma aplicação de Inteligência Artificial para ler e analisar documentos PDF, utilizando a API da OpenAI com controle de erros, validação de dados estruturados e gerenciamento seguro de chaves de acesso.

In [7]:
import os, time, json, types, unicodedata
from openai import OpenAI

def obter_chave(nome: str) -> str:
  """Le um segredo dos Secrets do Colab; fora do Colab, da variavel de ambiente."""
  try:
    from google.colab import userdata
    return userdata.get(nome)
  except ImportError:
    valor = os.getenv(nome)
    if not valor:
      raise RuntimeError(f"Defina {nome} nos Secrets do Colab ou no ambiente.")
    return valor

LLM_BASE_URL = "https://api.groq.com/openai/v1"
LLM_API_KEY = obter_chave("GROQ_API_KEY")
LLM_MODEL = "openai/gpt-oss-20b"

# Preco publicado em console.groq.com/pricing, conferido em 21/08/2026.
# Dolares por MILHAO de tokens.
PRECOS = {
    "openai/gpt-oss-20b":  {"entrada": 0.075, "saida": 0.30},
    "openai/gpt-oss-120b": {"entrada": 0.150, "saida": 0.60},
}
TPM = 8_000

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
print("chave carregada, termina em:", LLM_API_KEY[-4:])
print("modelo:", LLM_MODEL)
print(f"TPM do plano gratuito: {TPM:,}")

# 3. Função de conversa
def conversar(
    mensagem_do_usuario: str,
    instrucao_de_sistema: str | None = None,
    temperatura: float = 0.0,
) -> str:
    """Envia uma mensagem ao modelo e devolve o texto da resposta."""
    mensagens = []
    if instrucao_de_sistema:
        mensagens.append({"role": "system", "content": instrucao_de_sistema})
    mensagens.append({"role": "user", "content": mensagem_do_usuario})

    resposta = cliente.chat.completions.create(
        model=LLM_MODEL,
        messages=mensagens,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content or ""

# 4. Teste de Sanidade (Equivalente ao src/exemplo_chamada.py)
print("🔍 Testando conexão com a API...")
resposta_teste = conversar(
    mensagem_do_usuario="Responda em uma frase: O que é a Lei do Inquilinato?",
    instrucao_de_sistema="Seja conciso e responda em português do Brasil."
)
print("✅ Conexão bem-sucedida!")
print(f"🤖 Resposta do Modelo: {resposta_teste}")


chave carregada, termina em: WtUg
modelo: openai/gpt-oss-20b
TPM do plano gratuito: 8,000
🔍 Testando conexão com a API...
✅ Conexão bem-sucedida!
🤖 Resposta do Modelo: A Lei do Inquilinato (Lei nº 8.245/91) regula as relações locatícias de imóveis urbanos no Brasil, estabelecendo direitos e deveres de locadores e locatários.
